In [1]:
from transformers import BertTokenizer, BertForSequenceClassification

model_path = "/Users/chan/projects/models/bert-base-chinese"

/Users/chan/Documents/大模型资料库/LLM_dev/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path, num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /Users/chan/projects/models/bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
tokenizer

BertTokenizer(name_or_path='/Users/chan/projects/models/bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [4]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## 分词器

In [5]:
# 直接分词
tokenizer.tokenize("2026年欢迎来到黑马学习大模型")

['202', '##6', '年', '欢', '迎', '来', '到', '黑', '马', '学', '习', '大', '模', '型']

In [6]:
# 直接分词
tokenizer("2026年欢迎来到黑马学习大模型")

{'input_ids': [101, 9707, 8158, 2399, 3614, 6816, 3341, 1168, 7946, 7716, 2110, 739, 1920, 3563, 1798, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [7]:
# 句子对处理
# tokenizer(text="2026年欢迎来到黑马学习大模型", text_pair="新年好")
tokenizer("2026年欢迎来到黑马学习大模型", "新年好")

{'input_ids': [101, 9707, 8158, 2399, 3614, 6816, 3341, 1168, 7946, 7716, 2110, 739, 1920, 3563, 1798, 102, 3173, 2399, 1962, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [8]:
# 多个句子分词，返回pytorch 的tensor
tokenizer(["2026年欢迎来到黑马学习大模型", "AI算法"], return_tensors="pt", padding=True, truncation=True,
          max_length=None)

{'input_ids': tensor([[ 101, 9707, 8158, 2399, 3614, 6816, 3341, 1168, 7946, 7716, 2110,  739,
         1920, 3563, 1798,  102],
        [ 101,  100, 5050, 3791,  102,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

## 模型推理计算

In [9]:
inputs = "hello world！"

inputs = tokenizer(inputs, return_tensors='pt')

In [10]:
output = model(**inputs)

In [11]:
output

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.1685, -0.8365, -0.1640,  0.1725, -0.3013]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [12]:
logits = output.logits

In [13]:
# 取概率最大的标签，就是分类标签
logits.argmax()

tensor(3)

## MLM预测任务

In [16]:
from transformers import BertForMaskedLM

mlm = BertForMaskedLM.from_pretrained(model_path)

Some weights of the model checkpoint at /Users/chan/projects/models/bert-base-chinese were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [17]:
mlm

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [24]:
mlm_inputs = "下面是一条[MASK][MASK]的新闻，吴京是中国有名的武打明星"
mlm_inputs = tokenizer(mlm_inputs, return_tensors='pt')
mlm_inputs

{'input_ids': tensor([[ 101,  103,  776, 3221,  704, 1744, 3300, 1399, 4638, 3636, 2802, 3209,
         3215,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [25]:
output = mlm(**mlm_inputs)
logits = output.logits

In [26]:
logits.shape

torch.Size([1, 14, 21128])

In [27]:
index = 1  # [MASK] 位置在1 索引
logits[:, index, :]

tensor([[-7.9681, -8.3829, -8.5191,  ..., -8.3354, -6.2866, -6.3203]],
       grad_fn=<SliceBackward0>)

In [28]:
mlm_input_id = logits[:, index, :].argmax()
mlm_input_id

tensor(1426)

In [29]:
# "[MASK]京是中国的首都"
# [MASK] ===> 北
tokenizer.decode(mlm_input_id)

'吴'